In [50]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import re
import time

In [51]:
def set_up_driver():
    chrome_options = Options()
    #chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [52]:
longchau_categories_df = pd.read_csv(r'D:\OU\KhoaLuanTotNghiep\DrugRecommandation\crawlData_LongChau\longchau_categories.csv')
longchau_categories_df.head(3)

,Category Name,Category Link
0,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...
1,"Thuốc giải độc, khử độc và hỗ trợ cai nghiện\n...",https://nhathuoclongchau.com.vn/thuoc/thuoc-gi...
2,Thuốc da liễu\n286 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-da...


In [53]:
longchau_categories_df = longchau_categories_df.copy()

In [80]:
def get_medicine_links(driver, category_link):
    medicine_links = []
    try:
        driver.get(category_link)
        wait = WebDriverWait(driver, 10)
        time.sleep(3)

        # TÌm và bấm nút "Xem thêm" đến khi hết
        while True:
            try:
                # Scroll xuống dưới cùng trang để nút "Xem thêm" hiển thị
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)
                
                btn = wait.until(EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[contains(., 'Xem thêm') or contains(., 'Xem Thêm')]"
                )))
                # Scroll nút vào view trước khi click
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                time.sleep(1)
                btn.click()
                time.sleep(2)  
            except TimeoutException:
                # Nếu không tìm thấy nút, tức là hết dữ liệu
                break
            except Exception as e:
                # Exception khác cũng tính là hết dữ liệu
                break
        
        # Kéo về đầu trang để đảm bảo tất cả phần tử được tải
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)

        # Tìm vùng danh sách sản phẩm chính
        try:
            medicine_section = driver.find_element(By.ID, "category-page__products-section")
            # Tìm chỉ trong vùng này - các <a> với class chứa "p-3" và "block"
            medicine_links_elements = medicine_section.find_elements(
                By.XPATH, ".//a[@class and contains(@class, 'p-3') and contains(@class, 'block') and contains(@href, '/thuoc/')]"
            )
        except NoSuchElementException:
            # Fallback: nếu không tìm được section, tìm theo cấu trúc grid/flex
            medicine_links_elements = driver.find_elements(
                By.XPATH, "//div[contains(@class, 'grid') or contains(@class, 'flex')]//a[contains(@href, '/thuoc/') and contains(@class, 'block')]"
            )
        
        for element in medicine_links_elements:
            href = element.get_attribute("href")
            if href and '/thuoc/' in href:
                if not href.startswith('http'):
                    href = 'https://nhathuoclongchau.com.vn' + href
                medicine_links.append(href)

    except TimeoutException:
        print("Không tìm thấy các liên kết thuốc.")
    except Exception as e:
        print(f"Lỗi: {e}")

    return list(dict.fromkeys(medicine_links))

In [55]:
def crawl_medicine_data():
    driver = None
    all_medicine_links = []

    try:
        driver = set_up_driver()

        for _, row in longchau_categories_df.iterrows():
            category_name = row['Category Name']
            category_link = row['Category Link']
            print(f"Crawling category: {category_name} - {category_link}")

            medicine_links = get_medicine_links(driver, category_link)

            # Nếu medicine_links chỉ là list URL string
            for med_link in medicine_links:
                all_medicine_links.append((category_name, category_link, med_link))

            print(f"Đã thu thập {len(medicine_links)} liên kết thuốc từ danh mục này.")

    except Exception as e:
        print(f"Đã xảy ra lỗi: {e}")

    finally:
        if driver is not None:
            driver.quit()

    return pd.DataFrame(all_medicine_links, columns=['Category Name', 'Category Link', 'Medicine Link'])


In [62]:
df = crawl_medicine_data()
if isinstance(df, pd.DataFrame):
    print(f"Tổng số dòng: {len(df)}")
    display(df.head())

Crawling category: Thuốc dị ứng
138 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/thuoc-di-ung
Đã thu thập 157 liên kết thuốc từ danh mục này.
Crawling category: Thuốc giải độc, khử độc và hỗ trợ cai nghiện
7 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/thuoc-giai-doc-khu-doc-va-ho-tro-cai-nghien
Đã thu thập 8 liên kết thuốc từ danh mục này.
Crawling category: Thuốc da liễu
286 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/thuoc-da-lieu
Đã thu thập 336 liên kết thuốc từ danh mục này.
Crawling category: Miếng dán, cao xoa, dầu
53 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/mieng-dan-cao-xoa-dau
Đã thu thập 70 liên kết thuốc từ danh mục này.
Crawling category: Cơ - xương - khớp
176 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/co-xuong-khop
Đã thu thập 201 liên kết thuốc từ danh mục này.
Crawling category: Thuốc bổ & vitamin
282 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/thuoc-bo-and-vitamin
Đã thu thập 350 liên kết thuốc từ danh mục này.
Crawling category: Thuốc 

,Category Name,Category Link,Medicine Link
0,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/exopadin...
1,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/cetirizi...
2,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/allerpha...
3,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/histalon...
4,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/vomina-p...


In [63]:
df.to_csv("longchau_medicine_category_links(2).csv", index=False)

In [78]:
recrawl_links_list = [
    "https://nhathuoclongchau.com.vn/thuoc/thuoc-ho-hap"
]

def recrawl_medicine_links(recrawl_links_list):
    rows = []

    # map link -> category name (nếu có trong df)
    link_to_name = dict(zip(
        longchau_categories_df['Category Link'],
        longchau_categories_df['Category Name']
    ))

    driver = None
    try:
        driver = set_up_driver()

        for category_link in recrawl_links_list:
            category_name = link_to_name.get(category_link, "Unknown Category")
            print(f"Recrawling category: {category_name} - {category_link}")

            medicine_links = get_medicine_links(driver, category_link)
            
            rows.extend([(category_name, category_link, med_link) for med_link in medicine_links])

            print(f"Đã thu thập {len(medicine_links)} liên kết thuốc từ danh mục này.")

    except Exception as e:
        print(f"Đã xảy ra lỗi: {e}")

    finally:
        if driver is not None:
            driver.quit()

    return pd.DataFrame(rows, columns=['Category Name', 'Category Link', 'Medicine Link'])

In [81]:
df_recrawl_1 = recrawl_medicine_links(recrawl_links_list)
print(f"Số lượng link thu thập được sau khi recrawl: {len(df_recrawl_1)}")

Recrawling category: Thuốc hô hấp
310 sản phẩm - https://nhathuoclongchau.com.vn/thuoc/thuoc-ho-hap
Đã thu thập 372 liên kết thuốc từ danh mục này.
Số lượng link thu thập được sau khi recrawl: 372


In [82]:
# Bổ sung vào file csv đã lưu
df_old = pd.read_csv("longchau_medicine_category_links(2).csv")

# Merge và xóa duplicate dựa trên 'Medicine Link'
df_combined = pd.concat([df_old, df_recrawl_1], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['Medicine Link'], keep='first')

df_combined.to_csv("longchau_medicine_category_links(3).csv", index=False)

print(f"Tổng số link sau khi bổ sung: {len(df_combined)}")

Tổng số link sau khi bổ sung: 5760


In [84]:
df_category_final = pd.read_csv("longchau_medicine_category_links(3).csv")
df_category_final.shape
df_category_final.head()

,Category Name,Category Link,Medicine Link
0,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/exopadin...
1,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/cetirizi...
2,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/allerpha...
3,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/histalon...
4,Thuốc dị ứng\n138 sản phẩm,https://nhathuoclongchau.com.vn/thuoc/thuoc-di...,https://nhathuoclongchau.com.vn/thuoc/vomina-p...


In [61]:
# df_recrawl_2 = recrawl_medicine_links(set_up_driver(), recrawl_links_list[1])
# print(f"Số lượng link thu thập được sau khi recrawl: {len(df_recrawl_2)}")